# Bans around Twitch policy dates

The analysis ran locally with private course inputs. This preview shows selected aggregate results from that check. The course analysis cells below have had their original outputs and execution counts cleared.

## Verified results

| Comparison | Local verification |
| --- | --- |
| Original notebook | All 36 before/after rows agree to the displayed precision. |
| Submitted report | 11 of the 30 printed rows differ beyond rounding. |
| September 2023 | Two changes in bans per average viewer reach `p < 0.05` in the original tests. One is an increase and the other is a decrease. |

Two blocks were replayed using the daily audience series preserved in the course figures. The original audience files for those blocks are unavailable. The comparisons describe changes around the policy dates and do not establish their cause.

# Libraries

In [ ]:
from pathlib import Path
import os

# Jupyter may start the kernel in the notebook directory or in the repository root.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
DATA_DIR = Path(os.getenv("TWITCH_MODERATION_DATA") or repo_root / "data" / "private").expanduser()

import pandas as pd
import datetime as dt
import numpy as np
import matplotlib as plt
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import ttest_ind

# Defining the functions

### To prepare the datasets

In [ ]:
def preparation_dataframes(df_excel, df_ban, df_req):
    df_excel = df_excel.iloc[::-1].reset_index(drop=True)  # Inverting the dataset
    
    # Converting the date columns to datetime and localize timezone if necessary
    df_excel['Data'] = pd.to_datetime(df_excel['Data'])
    df_ban['created_at'] = pd.to_datetime(df_ban['created_at'])
    df_req['created_at'] = pd.to_datetime(df_req['created_at'])
    df_req['resolved_at'] = pd.to_datetime(df_req['resolved_at']).dt.tz_localize(None)  # Ensure timezone-naive datetime
    
    # Setting the new indexes
    df_excel.set_index('Data', inplace=True)
    df_ban.set_index('created_at', inplace=True)
    df_req.set_index('created_at', inplace=True)

    df_ban.index = df_ban.index.date
    df_req.index = df_req.index.date
    
    # Selecting and renaming the columns
    df_excel = df_excel[['Spettatori medi', 'Spettatori unici', 'Utenti chat']]
    df_excel.columns = ['Average viewers', 'Single viewers', 'Chat user']  # Rename columns
    df_ban = df_ban[['user_id', 'expires_at', 'reason', 'moderator_name']]
    df_req = df_req[['user_id', 'status', 'resolved_at']]
    
    # Count the number of banned people
    count_series = df_ban.index.to_series().groupby(df_ban.index).size()
    df_ban_count = count_series.to_frame(name='banned users')
    
    # Merge df_excel and df_ban
    combined_df = df_excel.join(df_ban_count, how='left')
    
    # Fill nan value with 0
    combined_df.fillna(0, inplace=True)
    
    # Dataframe with the ratios
    df_ratios = pd.DataFrame()
    df_ratios['Ratio Banned/average viewers'] = combined_df['banned users'] / combined_df['Average viewers']
    df_ratios['Ratio Banned/single viewers'] = combined_df['banned users'] / combined_df['Single viewers']
    df_ratios['Ratio Banned/chat user'] = combined_df['banned users'] / combined_df['Chat user']

    # In case of NaN they are turned into 0
    df_ratios.replace([float('inf'), -float('inf')], 0, inplace=True)
    df_ratios.fillna(0, inplace=True)
    
    return df_ban, df_req, combined_df, df_ratios


### To describe the data

In [ ]:
def describe(combined_df, df_ban, df_req):
    plot_columns = [col for col in combined_df.columns if col != 'key']
    colors = px.colors.qualitative.Plotly
    figures = []

    # Define a specific color map for status as per your requirements
    status_colors = {
        'pending': 'orange',  # Pending as orange
        'denied': 'blue',     # Denied as blue
        'acknowledged': 'red', # Acknowledged as red
        'canceled': 'purple',  # Canceled as purple
        'approved': 'green'   # Approved as green
    }

    # Time series plots for combined_df
    for i, column in enumerate(plot_columns):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=combined_df.index, y=combined_df[column], mode='lines', name=column, line=dict(color=colors[i % len(colors)])))
        fig.update_layout(title=f'Time Series of {column}', xaxis_title='Date', yaxis_title=column, xaxis=dict(rangeslider=dict(visible=True), type='date'))
        figures.append(fig)

    # Moderator name occurrences plot from df_ban
    if 'moderator_name' in df_ban.columns:
        moderator_count = df_ban['moderator_name'].value_counts()
        fig_moderator = px.bar(moderator_count, x=moderator_count.index, y=moderator_count.values, title='Occurrences of Each Moderator', labels={'x': 'Moderator Name', 'y': 'Count'})
        figures.append(fig_moderator)

    # Analysis of df_req showing time difference from request to resolution
    if 'resolved_at' in df_req.columns and 'status' in df_req.columns:
        df_req_index_as_datetime = pd.to_datetime(df_req.index)
        df_req['request_to_resolve_days'] = (df_req['resolved_at'] - df_req_index_as_datetime).dt.days
        df_req['hover_text'] = df_req_index_as_datetime.strftime('%Y-%m-%d')  # Custom hover text to show 'request_date'

        fig_req = px.scatter(df_req, x='resolved_at', y='request_to_resolve_days',
                             color='status', 
                             color_discrete_map=status_colors,  # Use the defined color map
                             hover_name='hover_text',
                             hover_data={'resolved_at': True, 'request_to_resolve_days': True, 'status': False},
                             title='Days from Request to Resolution',
                             labels={'resolved_at': 'Resolved Date', 'request_to_resolve_days': 'Days to Resolve'})
        fig_req.update_layout(xaxis=dict(rangeslider=dict(visible=True), type='date'))
        figures.append(fig_req)
        
    # Pie chart for the 'status' column in df_req with consistent colors
    if 'status' in df_req.columns:
        status_count = df_req['status'].value_counts()
        fig_status_pie = px.pie(status_count, values=status_count.values, names=status_count.index,
                                title='Distribution of Request Statuses',
                                color_discrete_map=status_colors)  # Use the same color map
        figures.append(fig_status_pie)

    return figures


### To compare dates and anomalies

In [ ]:
def bollinger_bands(df, column, window=20, width=4):
    # moving average
    df['MA'] = df[column].rolling(window=window).mean()

    # deviazione standard
    df['STD'] = df[column].rolling(window=window).std()

    # Bollinger bands
    df['Upper Band'] = df['MA'] + (df['STD'] * width)
    df['Lower Band'] = df['MA'] - (df['STD'] * width)

    # when it touch the upper and lower bands
    touches_upper = df[df[column] > df['Upper Band']]
    touches_lower = df[df[column] < df['Lower Band']]

    # policies of twitch
    important_dates = ['2020-03-05', '2021-01-22', '2023-09-01']

    # create the graph
    fig = go.Figure()

    # add the relationship
    fig.add_trace(go.Scatter(x=df.index, y=df[column], mode='lines', name=column))

    # add moving average
    fig.add_trace(go.Scatter(x=df.index, y=df['MA'], mode='lines', name='Moving Average'))

    # add Bollinger bands
    fig.add_trace(go.Scatter(x=df.index, y=df['Upper Band'], fill=None, mode='lines', line=dict(color='red'), name='Upper Bande'))
    fig.add_trace(go.Scatter(x=df.index, y=df['Lower Band'], fill=None, mode='lines', line=dict(color='green'), name='Lower Bande'))

    # add points when touch the lower or upper bands
    fig.add_trace(go.Scatter(x=touches_upper.index, y=touches_upper[column], mode='markers', marker=dict(color='red', size=10), name='Touch Upper Band'))
    fig.add_trace(go.Scatter(x=touches_lower.index, y=touches_lower[column], mode='markers', marker=dict(color='green', size=10), name='Touch Lower Band'))

    # Add policies of Twitch
    for date in important_dates:
        fig.add_vline(x=date, line=dict(color='gray', width=2, dash='dash'), line_width=2)

    # layout
    fig.update_layout(
        title=f"{column} 's graph with Bollinger Bands",
        xaxis_title='Data',
        yaxis_title=column,
        template='plotly_white',
        xaxis=dict(
            rangeslider=dict(
                visible=True
            ),
            type='date'
        )
    )
    
    return fig

In [ ]:
def policy_effect_analysis(df, column, policy_dates, window=30):

    results = []
    for date in policy_dates:
        policy_date = pd.to_datetime(date)
        pre_policy = df[(df.index < policy_date) & (df.index >= policy_date - pd.Timedelta(days=window))]
        post_policy = df[(df.index > policy_date) & (df.index <= policy_date + pd.Timedelta(days=window))]
        
        pre_mean = pre_policy[column].mean()
        post_mean = post_policy[column].mean()
        
        # Perform t-test
        t_stat, p_value = ttest_ind(pre_policy[column].dropna(), post_policy[column].dropna(), equal_var=False)
        
        # Append results
        results.append({
            'Policy Date': policy_date,
            'Pre-Policy Mean': pre_mean,
            'Post-Policy Mean': post_mean,
            'P-Value': p_value
        })
    
    return pd.DataFrame(results)

policy_dates = ['2020-03-05', '2021-01-22', '2023-09-01']


# Creator A

### Initialization

In [ ]:
df_excel = pd.read_csv(DATA_DIR / "policy/audience_a.csv")
df_ban = pd.read_excel(DATA_DIR / "policy/bans_a.xlsx")
df_req = pd.read_excel(DATA_DIR / "policy/appeals_a.xlsx")

### Creation of the datasets

In [ ]:
result_df = preparation_dataframes(df_excel, df_ban, df_req)
df_ban = result_df[0]
df_req = result_df[1]
df_combined = result_df[2]
df_ratios = result_df[3]

### Describing the data

In [ ]:
all_figures = describe(df_combined, df_ban, df_req)
for fig in all_figures:
     fig.show()

### Before/after comparisons and anomalies

In [ ]:
bollinger_bands(df_ratios, 'Ratio Banned/average viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/single viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/chat user').show()

In [ ]:
analysis_results1 = policy_effect_analysis(df_ratios, 'Ratio Banned/average viewers', policy_dates)
analysis_results2 = policy_effect_analysis(df_ratios, 'Ratio Banned/single viewers', policy_dates)
analysis_results3 = policy_effect_analysis(df_ratios, 'Ratio Banned/chat user', policy_dates)
print(analysis_results1, analysis_results2, analysis_results3)

# Creator B

### Initialization

In [ ]:
df_excel = pd.read_excel(DATA_DIR / "policy/audience_b.xlsx")
df_ban = pd.read_excel(DATA_DIR / "policy/bans_b.xlsx")
df_req = pd.read_excel(DATA_DIR / "policy/appeals_b.xlsx")

### Creation of the datasets

In [ ]:
result_df = preparation_dataframes(df_excel, df_ban, df_req)
df_ban = result_df[0]
df_req = result_df[1]
df_combined = result_df[2]
df_ratios = result_df[3]

### Describing the data

In [ ]:
all_figures = describe(df_combined, df_ban, df_req)
for fig in all_figures:
     fig.show()

### Before/after comparisons and anomalies

In [ ]:
bollinger_bands(df_ratios, 'Ratio Banned/average viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/single viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/chat user').show()

In [ ]:
analysis_results1 = policy_effect_analysis(df_ratios, 'Ratio Banned/average viewers', policy_dates)
analysis_results2 = policy_effect_analysis(df_ratios, 'Ratio Banned/single viewers', policy_dates)
analysis_results3 = policy_effect_analysis(df_ratios, 'Ratio Banned/chat user', policy_dates)
print(analysis_results1, analysis_results2, analysis_results3)

# Creator C

### Initialization

In [ ]:
df_excel = pd.read_excel(DATA_DIR / "policy/audience_c.xlsx")
df_ban = pd.read_excel(DATA_DIR / "policy/bans_c.xlsx")
df_req = pd.read_excel(DATA_DIR / "policy/appeals_c.xlsx")

### Creation of the datasets

In [ ]:
result_df = preparation_dataframes(df_excel, df_ban, df_req)
df_ban = result_df[0]
df_req = result_df[1]
df_combined = result_df[2]
df_ratios = result_df[3]

### Describing the data

In [ ]:
all_figures = describe(df_combined, df_ban, df_req)
for fig in all_figures:
     fig.show()

### Before/after comparisons and anomalies

In [ ]:
bollinger_bands(df_ratios, 'Ratio Banned/average viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/single viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/chat user').show()

In [ ]:
analysis_results1 = policy_effect_analysis(df_ratios, 'Ratio Banned/average viewers', policy_dates)
analysis_results2 = policy_effect_analysis(df_ratios, 'Ratio Banned/single viewers', policy_dates)
analysis_results3 = policy_effect_analysis(df_ratios, 'Ratio Banned/chat user', policy_dates)
print(analysis_results1, analysis_results2, analysis_results3)

# Creator D

### Initialization

In [ ]:
df_excel = pd.read_excel(DATA_DIR / "policy/audience_d.xlsx")
df_ban = pd.read_excel(DATA_DIR / "policy/bans_d.xlsx")
df_req = pd.read_excel(DATA_DIR / "policy/appeals_d.xlsx")

### Creation of the datasets

In [ ]:
result_df = preparation_dataframes(df_excel, df_ban, df_req)
df_ban = result_df[0]
df_req = result_df[1]
df_combined = result_df[2]
df_ratios = result_df[3]

### Describing the data

In [ ]:
all_figures = describe(df_combined, df_ban, df_req)
for fig in all_figures:
     fig.show()

### Before/after comparisons and anomalies

In [ ]:
bollinger_bands(df_ratios, 'Ratio Banned/average viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/single viewers').show()
bollinger_bands(df_ratios, 'Ratio Banned/chat user').show()

In [ ]:
analysis_results1 = policy_effect_analysis(df_ratios, 'Ratio Banned/average viewers', policy_dates)
analysis_results2 = policy_effect_analysis(df_ratios, 'Ratio Banned/single viewers', policy_dates)
analysis_results3 = policy_effect_analysis(df_ratios, 'Ratio Banned/chat user', policy_dates)
print(analysis_results1, analysis_results2, analysis_results3)